<a href="https://colab.research.google.com/github/ZainAfzalHashmi/Assignment-2/blob/main/Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.9 MB/s eta 0:00:00


In [4]:
import gradio as gr

def convert_units(category, from_unit, to_unit, value):
    """
    Converts a value from one unit to another within a given category.
    """
    try:
        value = float(value)
    except ValueError:
        return "Invalid input. Please enter a number."

    # --- Length Conversions (base unit: meters) ---
    length_factors = {
        "meters": 1.0,
        "kilometers": 1000.0,
        "centimeters": 0.01,
        "millimeters": 0.001,
        "miles": 1609.34,
        "yards": 0.9144,
        "feet": 0.3048,
        "inches": 0.0254,
    }

    # --- Weight/Mass Conversions (base unit: kilograms) ---
    weight_factors = {
        "kilograms": 1.0,
        "grams": 0.001,
        "milligrams": 0.000001,
        "pounds": 0.453592,
        "ounces": 0.0283495,
    }

    # --- Temperature Conversions (special handling, no single base factor) ---
    def convert_temperature(from_unit, to_unit, temp_val):
        if from_unit == to_unit:
            return temp_val

        # Convert to Celsius first
        if from_unit == "celsius":
            celsius = temp_val
        elif from_unit == "fahrenheit":
            celsius = (temp_val - 32) * 5/9
        elif from_unit == "kelvin":
            celsius = temp_val - 273.15
        else:
            return "Invalid 'from' temperature unit."

        # Convert from Celsius to target unit
        if to_unit == "celsius":
            return celsius
        elif to_unit == "fahrenheit":
            return (celsius * 9/5) + 32
        elif to_unit == "kelvin":
            return celsius + 273.15
        else:
            return "Invalid 'to' temperature unit."

    # --- Perform Conversion ---
    if category == "Length":
        if from_unit not in length_factors or to_unit not in length_factors:
            return "Invalid length unit selected."
        # Convert to base unit (meters) then to target unit
        value_in_meters = value * length_factors[from_unit]
        result = value_in_meters / length_factors[to_unit]
        return f"{result:.4f} {to_unit}" # Format to 4 decimal places
    elif category == "Weight/Mass":
        if from_unit not in weight_factors or to_unit not in weight_factors:
            return "Invalid weight unit selected."
        # Convert to base unit (kilograms) then to target unit
        value_in_kg = value * weight_factors[from_unit]
        result = value_in_kg / weight_factors[to_unit]
        return f"{result:.4f} {to_unit}"
    elif category == "Temperature":
        result = convert_temperature(from_unit, to_unit, value)
        return f"{result:.2f} {to_unit}" # Temperature usually needs fewer decimal places
    else:
        return "Please select a conversion category."

In [7]:

# Define unit options for each category
unit_options = {
    "Length": ["meters", "kilometers", "centimeters", "millimeters", "miles", "yards", "feet", "inches"],
    "Weight/Mass": ["kilograms", "grams", "milligrams", "pounds", "ounces"],
    "Temperature": ["celsius", "fahrenheit", "kelvin"],
}

# Gradio Interface setup
with gr.Blocks(title="Versatile Unit Converter") as demo:
    gr.Markdown(
        """
        # 🚀 Versatile Unit Converter
        Easily convert between various units of length, weight, and temperature!
        """
    )

    with gr.Row():
        category_dropdown = gr.Dropdown(
            choices=list(unit_options.keys()),
            label="Conversion Category",
            value="Length",  # Default value
            interactive=True
        )

    with gr.Row():
        from_unit_dropdown = gr.Dropdown(
            choices=unit_options["Length"],  # Initial choices based on default category
            label="From Unit",
            value="meters",
            interactive=True
        )
        to_unit_dropdown = gr.Dropdown(
            choices=unit_options["Length"],  # Initial choices based on default category
            label="To Unit",
            value="kilometers",
            interactive=True
        )

    with gr.Row():
        value_input = gr.Number(label="Value to Convert", value=1.0)
        output_text = gr.Textbox(label="Converted Result", interactive=False)

    convert_button = gr.Button("Convert")

    # --- UI Logic ---
    # Update 'from' and 'to' unit dropdowns when category changes
    def update_unit_dropdowns(category):
        return gr.Dropdown(choices=unit_options[category]), gr.Dropdown(choices=unit_options[category])

    category_dropdown.change(
        update_unit_dropdowns,
        inputs=category_dropdown,
        outputs=[from_unit_dropdown, to_unit_dropdown]
    )

    # Convert on button click
    convert_button.click(
        fn=convert_units,
        inputs=[category_dropdown, from_unit_dropdown, to_unit_dropdown, value_input],
        outputs=output_text
    )

    # Convert on input change (optional, but good for responsiveness)
    value_input.change(
        fn=convert_units,
        inputs=[category_dropdown, from_unit_dropdown, to_unit_dropdown, value_input],
        outputs=output_text
    )
    from_unit_dropdown.change(
        fn=convert_units,
        inputs=[category_dropdown, from_unit_dropdown, to_unit_dropdown, value_input],
        outputs=output_text
    )
    to_unit_dropdown.change(
        fn=convert_units,
        inputs=[category_dropdown, from_unit_dropdown, to_unit_dropdown, value_input],
        outputs=output_text
    )

In [8]:
demo.launch(share=True) # share=True generates a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4fc04167e568861758.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
